![Databricks Academy](https://files.training.databricks.com/binder/prod_main/get-started-with-databricks-end-to-end-en_us-1.0.0/images/20260922T145855Z/Get Started with Databricks End to End/Includes/images/databricks_academy_logo.png)

# Demo - Tracking and managing models with MLflow

In this demo, we will explore the model development lifecycle using MLflow with bakehouse sales data. You'll train a model to predict product purchases, log artifacts, metrics, and parameters using MLflow, and then register the model in Unity Catalog for production readiness. 

You don't need to be a data scientist to follow along. This lesson shows you what model training looks like in Databricks so you can recognize the pattern when working with data science teams.

**Learning Objectives:**

- Prepare features using SQL from bakehouse sales transactions
- Train and log models using MLflow
- Track experiments and view runs using MLflow UI
- Register a model in Unity Catalog
- Use model version aliases like `staging` and `dev`

## REQUIRED - Select a Compute Environment

<div style="border-left: 4px solid #FF5F46; background: rgba(255,95,70,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#98102A; margin-bottom:6px; font-size:15pt;">Select Serverless Compute</strong>
  <div style="color:#0b2026; font-size:14pt; line-height:1.55;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](https://files.training.databricks.com/binder/prod_main/get-started-with-databricks-end-to-end-en_us-1.0.0/images/20260922T145855Z/Get Started with Databricks End to End/Includes/images/select-compute.png)

<br></br>
How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:** This notebook requires **SQL and Python**, and was tested using **Serverless V5**. All-purpose compute may work but is not guaranteed to behave the same or support all features demonstrated.

  </div>
</div>


## A. Key Terms

Before you begin, review the definitions for the machine learning concepts used in this lesson.

| Term | Definition |
| --- | --- |
| **Features** | Input variables used to make predictions. In this case, these are customer behaviors like transaction count, average purchase amount, and payment preferences. |
| **Train a model** | The process of teaching a machine learning algorithm to recognize patterns in data. The model learns by analyzing examples (training data) to make predictions on new data. |
| **MLflow** | An open-source platform for managing the complete machine learning lifecycle, including experiment tracking, model versioning, and deployment. |
| **Experiments** | A collection of related model training runs, which organizes all attempts to solve a specific prediction problem. |
| **Runs** | A single execution of model training code. Each run captures parameters (settings), metrics (performance scores), and artifacts (the trained model files). |
| **Register a model** | The process of saving a trained model to a centralized registry (Unity Catalog) where it can be versioned, shared, and deployed. |
| **Model version aliases** | Human-readable labels (like `dev`, `staging`, `production`) that point to specific model versions, making it easier to manage which model is deployed where. |

## B. Classroom Setup

Run the cell below to prepare your environment. This setup script will:

1. Create or use your personal catalog named **labuser_UNIQUE_ID**.
2. Create a schema called **ai_ml_ds**.
3. Set your default catalog and schema.

In [0]:
%run ../Includes/Classroom-Setup-ai-ml-ds

## C. Data Preparation with SQL

In this first part, you will use SQL to read and prepare **features** from the bakehouse sales transactions that will be used for building your machine learning model later. You will create features that help predict which product a customer will purchase based on:

- Customer purchase history (total transactions, average quantity)
- Franchise characteristics
- Transaction timing patterns
- Payment behavior

In [0]:
%sql
-- Create features from sales transactions
-- We'll aggregate customer behavior to predict their next product purchase
CREATE OR REPLACE TABLE customer_features AS
SELECT 
  customerID,
  franchiseID,
  product,
  -- Purchase behavior features
  COUNT(*) as transaction_count,
  AVG(quantity) as avg_quantity,
  AVG(totalPrice) as avg_total_price,
  SUM(totalPrice) as total_spent,
  -- Payment preference (encode as numbers)
  SUM(CASE WHEN paymentMethod = 'visa' THEN 1 ELSE 0 END) as visa_count,
  SUM(CASE WHEN paymentMethod = 'mastercard' THEN 1 ELSE 0 END) as mastercard_count,
  SUM(CASE WHEN paymentMethod = 'amex' THEN 1 ELSE 0 END) as amex_count,
  -- Time-based features
  HOUR(MAX(dateTime)) as last_purchase_hour,
  DAYOFWEEK(MAX(dateTime)) as last_purchase_day
FROM samples.bakehouse.sales_transactions
GROUP BY customerID, franchiseID, product;

SELECT * FROM customer_features LIMIT 10

### C1. Prepare Training Data

Now that you've created a feature table using SQL, you need to prepare the data for machine learning:

1. **Convert to Pandas** - Transform the SQL results into a Pandas DataFrame for scikit-learn compatibility.
2. **Encode the target** - Convert product names (text) into numeric labels that the model can predict.
3. **Select features** - Choose which columns will be used as input features for prediction.
4. **Split the data** - Divide into training (80%) and testing (20%) sets to evaluate model performance.

This step transforms your business data into the format required for machine learning model training.

**NOTE:** You convert the Spark DataFrame to a Pandas DataFrame (using `.toPandas()`) because the ML library used in this example, scikit-learn, works with Pandas DataFrames.

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Convert the SQL results to Pandas DataFrame
training_df = spark.sql("SELECT * FROM customer_features").toPandas()

# Encode the target variable (product) as numeric
le = LabelEncoder()
training_df['product_encoded'] = le.fit_transform(training_df['product'])

# Select features and target
feature_cols = ['franchiseID', 'transaction_count', 'avg_quantity', 'avg_total_price', 
                'total_spent', 'visa_count', 'mastercard_count', 'amex_count',
                'last_purchase_hour', 'last_purchase_day']

X = training_df[feature_cols]
y = training_df['product_encoded']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Products to predict: {list(le.classes_)}")

In [0]:
# Display the training data with features and target
display(training_df[feature_cols + ['product', 'product_encoded']].head(20))

## D. Walkthrough of the Run Block

The panel below breaks down the key lines of the training run. **Click a highlighted line or an annotation chip** to reveal an explanation of what that line contributes to the tracked run.

<div id="sx-mount"></div>

<style>
  .sx-wrap { position: relative; font-family: sans-serif; border: 1px solid #DCE0E2; border-radius: 12px; overflow: hidden; background: #F9F7F4; margin: 16px 0; }
  .sx-title { background: #EEEDE9; border-bottom: 1px solid #DCE0E2; padding: 8px 14px; font-size: 14pt; font-weight: 700; color: #5A6F77; }
  .sx-rail { display: flex; flex-wrap: wrap; gap: 8px; padding: 10px 14px; background: #F8F9FC; border-bottom: 1px solid #EEEDE9; }
  .sx-chip { border: 1px solid #DCE0E2; background: #FFFFFF; border-radius: 999px; padding: 5px 12px; font-size: 14pt; font-weight: 700; color: #1B3139; cursor: pointer; }
  .sx-chip:hover, .sx-chip.active { background: rgba(66,153,224,0.12); border-color: #4299E0; color: #2272B4; }
  .sx-chip-hovered-from-line { background: rgba(66,153,224,0.12); border-color: #4299E0; color: #2272B4; }
  .sx-code { padding: 10px 0; overflow-x: auto; font-size: 14pt; line-height: 1.55; }
  .sx-code, .sx-line-code { font-family: Menlo, Consolas, monospace !important; }
  .sx-line { display: grid; grid-template-columns: 52px 1fr; }
  .sx-line.no-lines { grid-template-columns: 1fr; }
  .sx-line.no-lines .sx-line-code { padding-left: 16px; }
  .sx-line-no { color: #5A6F77; text-align: right; padding: 0 12px; user-select: none; border-right: 1px solid #EEEDE9; }
  .sx-line-code { padding: 0 16px; white-space: pre; }
  .sx-active-line { background: rgba(66,153,224,0.18); }
  .sx-active-line .sx-line-code { border-left: 3px solid #4299E0; padding-left: 13px; }
  .sx-line.sx-active-line { background: rgba(66,153,224,0.18); }
  .sx-line.sx-active-line .sx-line-code { border-left: 3px solid #4299E0; padding-left: 13px; }
  .sx-hotspot { background: rgba(66,153,224,0.08); cursor: pointer; }
  .sx-hotspot .sx-line-code { border-left: 3px solid rgba(66,153,224,0.5); padding-left: 13px; }
  .sx-hotspot:hover, .sx-hovered-from-chip { background: rgba(66,153,224,0.16); }
  .sx-line.sx-hotspot.sx-active-line { background: rgba(66,153,224,0.22) !important; }
  .sx-line.sx-hotspot.sx-active-line .sx-line-code { border-left: 3px solid #4299E0 !important; padding-left: 13px !important; }
  .sx-hotspot .sx-line-no { color: #2272B4; font-weight: 700; }
  .sx-overlay { display: none; position: absolute; right: 18px; top: 96px; width: 360px; max-width: calc(100% - 36px); background: #FFFFFF; border: 1px solid #DCE0E2; border-left: 5px solid #4299E0; border-radius: 12px; box-shadow: 0 6px 18px rgba(27,49,57,0.12), 0 16px 36px rgba(27,49,57,0.10); padding: 14px 16px; z-index: 5; }
  .sx-overlay-title { font-weight: 800; font-size: 15pt; margin-bottom: 6px; color: #1B3139; }
  .sx-overlay-body { color: #0b2026; line-height: 1.45; font-size: 14pt; }
</style>

<script>
(function () {
  const config = {
    mountId: "sx-mount",
    title: "MLflow Breakdown",
    showAnnotationRail: true,
    hotspots: [
      {
        title: "Start the run",
        startLine: 1,
        endLine: 1,
        description: "<code>mlflow.start_run()</code> opens a named run inside the active experiment. Everything logged within this <code>with</code> block is attached to this single run, and the <em>run object</em> exposes metadata such as <code>run.info.run_id</code>."
      },
      {
        title: "Enable autolog",
        startLine: 7,
        endLine: 10,
        description: "<code>mlflow.sklearn.autolog()</code> automatically captures parameters, metrics, and the model. It <em>must</em> be called before <code>fit()</code> so training-phase values are recorded. <code>log_input_examples=True</code> stores sample inputs and <code>silent=True</code> suppresses autolog warnings."
      },
      {
        title: "Fit the model",
        startLine: 13,
        endLine: 13,
        description: "Training the estimator with <code>fit()</code> triggers autologging. Parameters like <code>n_estimators</code> and training metrics are logged to the run without any explicit logging calls."
      },
      {
        title: "Log custom metric",
        startLine: 19,
        endLine: 19,
        description: "<code>mlflow.log_metric()</code> records a metric autolog does not compute — here the macro-averaged test F1 score. Custom metrics complement autologged values for a fuller picture of model quality."
      },
      {
        title: "Log the model",
        startLine: 21,
        endLine: 26,
        description: "<code>mlflow.sklearn.log_model()</code> saves the trained model as a versioned artifact. The <code>signature</code> from <code>infer_signature()</code> records the input/output schema and <code>input_example</code> stores a sample payload for validation and serving."
      },
      {
        title: "Build the model URI",
        startLine: 28,
        endLine: 28,
        description: "The <code>runs:/&lt;run_id&gt;/model-artifacts</code> URI uniquely references the logged model within this run. Use it to load the model later or to register it in the Model Registry."
      }
    ],
    source: `with mlflow.start_run(run_name='bakehouse-product-prediction-run') as run:
    # Initialize the Random Forest classifier
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

    # Enable automatic logging of inputs, metrics, parameters, and models
    # Must be called BEFORE fit() to capture training-phase values
    mlflow.sklearn.autolog(
        log_input_examples=True,
        silent=True
    )

    # Fit the model on the training data
    rf_classifier.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = rf_classifier.predict(X_test)

    # Calculate F1 score with 'macro' averaging for multiclass
    mlflow.log_metric("test_f1", f1_score(y_test, y_pred, average='macro'))

    mlflow.sklearn.log_model(
        rf_classifier,
        artifact_path="model-artifacts",
        input_example=X_train[:3],
        signature=infer_signature(X_train, y_train)
    )

    model_uri = f"runs:/{run.info.run_id}/model-artifacts"`,
    showLineNumbers: true,
    displayStartLine: 1,
  };

  function escapeHtml(value) {
    return String(value)
      .replace(/&/g, "&amp;")
      .replace(/</g, "&lt;")
      .replace(/>/g, "&gt;");
  }

  function renderSourceExplorer(config) {
    const mount = document.getElementById(config.mountId);
    const lines = config.source.replace(/\n$/, "").split("\n");
    const start = config.startLine || 1;
    const end = config.endLine || lines.length;
    const visible = lines.slice(start - 1, end);
    const hotspots = config.hotspots || [];
    const displayStartLine = config.displayStartLine || start;

    function hotspotFor(lineNo) {
      const index = hotspots.findIndex(h => lineNo >= h.startLine && lineNo <= h.endLine);
      return index >= 0 ? { hotspot: hotspots[index], index } : null;
    }

    const codeHtml = visible.map((line, idx) => {
      const actualLineNo = start + idx;
      const displayLineNo = displayStartLine + idx;
      const hotspot = hotspotFor(actualLineNo);

      const classes = hotspot
        ? `sx-line sx-hotspot sx-hotspot-${hotspot.index}`
        : "sx-line";

      const attrs = hotspot
        ? `data-hotspot-index="${hotspot.index}"`
        : "";

      return `
        <div class="${classes} ${config.showLineNumbers ? "" : "no-lines"}" ${attrs}>
          ${config.showLineNumbers
            ? `<div class="sx-line-no">${displayLineNo}</div>`
            : ""}
          <div class="sx-line-code">${escapeHtml(line)}</div>
        </div>`;
    }).join("");

    const railHtml = config.showAnnotationRail && hotspots.length > 0
      ? `<div class="sx-rail">
          ${hotspots.map((h, i) => `
            <button class="sx-chip" data-hotspot-index="${i}">
              ${escapeHtml(h.title)}
            </button>
          `).join("")}
        </div>`
      : "";

    const overlayHtml = hotspots.length > 0
      ? `<div class="sx-overlay">
          <div class="sx-overlay-title"></div>
          <div class="sx-overlay-body"></div>
        </div>`
      : "";

    mount.innerHTML = `
      <div class="sx-wrap">
        <div class="sx-title">${escapeHtml(config.title)}</div>
        ${railHtml}
        <div class="sx-code">${codeHtml}</div>
        ${overlayHtml}
      </div>`;

    if (hotspots.length === 0) return;

    const overlay = mount.querySelector(".sx-overlay");
    const overlayTitle = mount.querySelector(".sx-overlay-title");
    const overlayBody = mount.querySelector(".sx-overlay-body");
    let activeHotspotIndex = null;

    function setActiveLines(idx) {
      mount.querySelectorAll(".sx-active-line").forEach(el => {
        el.classList.remove("sx-active-line");
      });
      if (idx === null) return;
      mount.querySelectorAll(`.sx-hotspot-${idx}`).forEach(el => {
        el.classList.add("sx-active-line");
      });
    }

    function setActiveChip(idx) {
      mount.querySelectorAll(".sx-chip").forEach(chip => {
        chip.classList.toggle(
          "active",
          idx !== null && Number(chip.dataset.hotspotIndex) === idx
        );
      });
    }

    function setChipHover(idx, on) {
      mount.querySelectorAll(".sx-chip").forEach(chip => {
        const isMatch = Number(chip.dataset.hotspotIndex) === idx;
        chip.classList.toggle("sx-chip-hovered-from-line", on && isMatch);
      });
    }

    function setLineHover(idx, on) {
      mount.querySelectorAll(`.sx-hotspot-${idx}`).forEach(line => {
        line.classList.toggle("sx-hovered-from-chip", on);
      });
    }

    function showHotspot(idx, triggerEl = null) {
      const hotspot = hotspots[idx];
      if (!hotspot) return;

      if (overlay.style.display === "block" && activeHotspotIndex === idx) {
        overlay.style.display = "none";
        activeHotspotIndex = null;
        setActiveChip(null);
        setActiveLines(null);
        return;
      }

      overlayTitle.textContent = hotspot.title;
      overlayBody.innerHTML = hotspot.description;
      overlay.style.display = "block";

      const wrap = mount.querySelector(".sx-wrap");
      const trigger = triggerEl || mount.querySelector(`.sx-hotspot-${idx}`) || mount.querySelector(`.sx-chip[data-hotspot-index="${idx}"]`);

      if (trigger) {
        const wrapRect = wrap.getBoundingClientRect();
        const triggerRect = trigger.getBoundingClientRect();
        const desiredTop = triggerRect.top - wrapRect.top;
        const maxTop = wrap.clientHeight - overlay.offsetHeight - 18;
        overlay.style.top = `${Math.max(18, Math.min(desiredTop, maxTop))}px`;
      }

      activeHotspotIndex = idx;
      setActiveChip(idx);
      setActiveLines(idx);
    }

    mount.querySelectorAll(".sx-hotspot").forEach(el => {
      const idx = Number(el.dataset.hotspotIndex);
      el.addEventListener("click", () => { showHotspot(idx, el); });
      el.addEventListener("mouseenter", () => { setLineHover(idx, true); setChipHover(idx, true); });
      el.addEventListener("mouseleave", () => {
        setLineHover(idx, false);
        setChipHover(idx, false);
        if (activeHotspotIndex !== null) { setActiveLines(activeHotspotIndex); }
      });
    });

    mount.querySelectorAll(".sx-chip").forEach(chip => {
      const idx = Number(chip.dataset.hotspotIndex);
      chip.addEventListener("click", () => {
        const associatedLine = mount.querySelector(`.sx-hotspot-${idx}`);
        showHotspot(idx, associatedLine || chip);
      });
      chip.addEventListener("mouseenter", () => { setLineHover(idx, true); });
      chip.addEventListener("mouseleave", () => {
        setLineHover(idx, false);
        if (activeHotspotIndex !== null) { setActiveLines(activeHotspotIndex); }
      });
    });

    mount.querySelector(".sx-wrap").addEventListener("click", (e) => {
      if (e.target.closest(".sx-hotspot")) return;
      if (e.target.closest(".sx-chip")) return;
      if (e.target.closest(".sx-overlay")) return;
      overlay.style.display = "none";
      activeHotspotIndex = null;
      setActiveChip(null);
      setActiveLines(null);
    });
  }

  renderSourceExplorer(config);
})();
</script>

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# set the path for mlflow experiment
mlflow.set_experiment(f"/Users/{username}/bakehouse-product-prediction-experiment")

with mlflow.start_run(run_name = 'bakehouse-product-prediction-run') as run:  
    # Initialize the Random Forest classifier
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

    # Enable automatic logging of input samples, metrics, parameters, and models
    # Must be called BEFORE fit() to capture training-phase metrics and parameters
    mlflow.sklearn.autolog(
        log_input_examples = True,
        silent = True
    )

    # Fit the model on the training data
    rf_classifier.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = rf_classifier.predict(X_test)

    # Calculate F1 score with 'macro' averaging for multiclass
    mlflow.log_metric("test_f1", f1_score(y_test, y_pred, average='macro'))
        
    mlflow.sklearn.log_model(
        rf_classifier,
        name = "model-artifacts", 
        input_example=X_train[:3],
        signature=infer_signature(X_train, y_train)
    )

    model_uri = f"runs:/{run.info.run_id}/model-artifacts"
    
print(f"Model trained successfully! Run ID: {run.info.run_id}")

### D1. Viewing MLflow Run Details

Now that you've logged your model training run, here's how to view and explore the details.

#### D1.1 Accessing the MLflow UI

You can open the MLflow UI in one of two ways.

**Option 1: Click the Experiment Link**

1. In the cell output above, locate the link that says **View Logged Model at:...**.
2. Click this link to open the MLflow UI directly.

**Option 2: Navigate via Experiments**

1. Click **Experiments** in the left sidebar.
2. Find your experiment: `bakehouse-product-prediction-experiment`.
3. Click on the run name: `bakehouse-product-prediction-run`.

#### D1.2 What You'll See in the Run Details

The MLflow UI displays comprehensive information about your training run across several sections.

**Metrics**

- `test_f1` — the F1 score you logged manually on the test set.
- Additional metrics captured by `autolog` (e.g. training accuracy, precision, recall).

**Parameters**

- All hyperparameters captured by `autolog` (e.g. `n_estimators`, `random_state`, `max_depth`).

**About this logged model**

- Meta information displayed on the right panel: creation time, author, experiment name, and run ID.

**Artifacts tab**

- The pickled model file and Python environment configuration — these are required when loading the model later.
- Under **Source run artifacts** you will find additional files generated by `autolog`, including a training confusion matrix and a feature importance chart.

#### D1.3 Why This Matters

Tracking run details supports several key parts of the machine learning workflow.

- **Reproducibility** - Every detail needed to recreate this model is saved.
- **Comparison** - Run multiple experiments and compare metrics side-by-side.
- **Collaboration** - Share run links with teammates for review.
- **Debugging** - Review parameters and metrics when investigating issues.
- **Production** - Track which model version is deployed and performing best.

After reviewing the run, you can compare it with other experiments, load the model for predictions with `mlflow.sklearn.load_model(model_uri)`, deploy the model to a serving endpoint, or set up model monitoring and alerts.

## E. Register the Model to the Model Registry

Now that you've trained and logged your model, the next step is to **register it to Unity Catalog** under your catalog and schema. This makes the model available for:

- **Deployment** - Move models from development to staging to production.
- **Version Control** - Track multiple versions of the same model.
- **Collaboration** - Share models across teams and projects.
- **Discovery** - Find and reuse models across the organization.
- **Lineage** - Automatically tracks the upstream tables used for training and downstream consumers such as serving endpoints.


### E1. What Registration Does

1. **Creates a named model** - Assigns a user-friendly name to your model.
2. **Versions automatically** - Each registration creates a new version (1, 2, 3...).
3. **Links to the run** - Maintains connection to training metrics and parameters.
4. **Enables lifecycle management** - Supports aliases like `dev`, `staging`, and `production`.
5. **Captures lineage** - Unity Catalog records the relationship between training data, the MLflow run, and the registered model automatically.

After registration, the model can be loaded by name and version from any notebook, job, or serving endpoint.

In [0]:
# Register the model to Unity Catalog
# Define the model name in Unity Catalog format: catalog.schema.model_name
model_name = f"{my_catalog}.ai_ml_ds.bakehouse_product_predictor_{username.split('@')[0].replace('.', '_')}"

# Register the model in Unity Catalog
registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)

print(f"Model registered as: {model_name}")
print(f"Version: {registered_model.version}")

   
### E2. Model Lineage

One of the most valuable features of registering models to Unity Catalog is **automatic lineage** — no extra configuration required. Unity Catalog builds a lineage graph connecting your model to the data it was trained on and every downstream consumer that uses it.

**What lineage tracks**

- **Upstream** — the tables read during training (e.g. `bakehouse_sales`), the MLflow run that produced the model, and the notebook or job that triggered it.
- **Downstream** — serving endpoints, notebooks, and jobs that load the registered model by name or alias.

**Where to find it**

1. Click **Catalog** in the left sidebar.
2. Navigate to your model under `catalog_name` > `schema_name` > **Models**.
3. Open the model and select the **Lineage** tab.

The lineage graph lets you trace exactly which data produced a given model version and which systems depend on it, which is critical for governance, debugging model quality issues, and assessing the impact of changes to upstream tables.

## F. Model Aliases: Managing Deployment Stages

Now that your model is registered, you'll add an **alias** to this version. Aliases are human-readable labels that point to specific model versions, making deployment management much easier.

### F1. What Are Model Aliases?

Model aliases are named pointers to specific model versions. Instead of referencing `bakehouse_product_predictor version 3`, you can reference `bakehouse_product_predictor@production`.

The following table summarizes common alias patterns.

| Alias | Purpose |
| --- | --- |
| `dev` | Latest model being developed and tested by data scientists. |
| `staging` | Model undergoing QA testing before production release. |
| `production` | Currently serving live predictions to end users. |
| `champion` | Best performing model (for A/B testing scenarios). |
| `challenger` | New model being compared against the champion. |

   
### F2. Example Workflow

A typical alias-driven deployment workflow follows these steps:

1. Train a new model and register it as version 3.
2. Assign the `dev` alias and test in the development environment.
3. When tests pass, move the `staging` alias to version 3.
4. When QA approves, move the `production` alias to version 3.
5. Complete all of this without changing any application code.

In [0]:
from mlflow.tracking.client import MlflowClient

# Initialize an MLflow Client
client = MlflowClient()

# Assign a "dev" alias to model version 1
client.set_registered_model_alias(
    name= registered_model.name,  # The registered model name
    alias="dev",  # The alias representing the dev environment
    version=registered_model.version  # The version of the model you want to move to "dev"
)

### F3. Loading Models with Aliases

The two approaches below show how you can reference a model at load time. **Click a highlighted line or an annotation chip** to compare the version-pinned and alias-based approaches.

<div id="g3-mount"></div>

<style>
  .sx-wrap { position: relative; font-family: sans-serif; border: 1px solid #DCE0E2; border-radius: 12px; overflow: hidden; background: #F9F7F4; margin: 16px 0; }
  .sx-title { background: #EEEDE9; border-bottom: 1px solid #DCE0E2; padding: 8px 14px; font-size: 14pt; font-weight: 700; color: #5A6F77; }
  .sx-rail { display: flex; flex-wrap: wrap; gap: 8px; padding: 10px 14px; background: #F8F9FC; border-bottom: 1px solid #EEEDE9; }
  .sx-chip { border: 1px solid #DCE0E2; background: #FFFFFF; border-radius: 999px; padding: 5px 12px; font-size: 14pt; font-weight: 700; color: #1B3139; cursor: pointer; }
  .sx-chip:hover, .sx-chip.active { background: rgba(66,153,224,0.12); border-color: #4299E0; color: #2272B4; }
  .sx-chip-hovered-from-line { background: rgba(66,153,224,0.12); border-color: #4299E0; color: #2272B4; }
  .sx-code { padding: 10px 0; overflow-x: auto; font-size: 14pt; line-height: 1.55; }
  .sx-code, .sx-line-code { font-family: Menlo, Consolas, monospace !important; }
  .sx-line { display: grid; grid-template-columns: 52px 1fr; }
  .sx-line.no-lines { grid-template-columns: 1fr; }
  .sx-line.no-lines .sx-line-code { padding-left: 16px; }
  .sx-line-no { color: #5A6F77; text-align: right; padding: 0 12px; user-select: none; border-right: 1px solid #EEEDE9; }
  .sx-line-code { padding: 0 16px; white-space: pre; }
  .sx-active-line { background: rgba(66,153,224,0.18); }
  .sx-active-line .sx-line-code { border-left: 3px solid #4299E0; padding-left: 13px; }
  .sx-line.sx-active-line { background: rgba(66,153,224,0.18); }
  .sx-line.sx-active-line .sx-line-code { border-left: 3px solid #4299E0; padding-left: 13px; }
  .sx-hotspot { background: rgba(66,153,224,0.08); cursor: pointer; }
  .sx-hotspot .sx-line-code { border-left: 3px solid rgba(66,153,224,0.5); padding-left: 13px; }
  .sx-hotspot:hover, .sx-hovered-from-chip { background: rgba(66,153,224,0.16); }
  .sx-line.sx-hotspot.sx-active-line { background: rgba(66,153,224,0.22) !important; }
  .sx-line.sx-hotspot.sx-active-line .sx-line-code { border-left: 3px solid #4299E0 !important; padding-left: 13px !important; }
  .sx-hotspot .sx-line-no { color: #2272B4; font-weight: 700; }
  .sx-overlay { display: none; position: absolute; right: 18px; top: 96px; width: 360px; max-width: calc(100% - 36px); background: #FFFFFF; border: 1px solid #DCE0E2; border-left: 5px solid #4299E0; border-radius: 12px; box-shadow: 0 6px 18px rgba(27,49,57,0.12), 0 16px 36px rgba(27,49,57,0.10); padding: 14px 16px; z-index: 5; }
  .sx-overlay-title { font-weight: 800; font-size: 15pt; margin-bottom: 6px; color: #1B3139; }
  .sx-overlay-body { color: #0b2026; line-height: 1.45; font-size: 14pt; }
  #g3-mount .sx-overlay { position: static; width: auto; max-width: none; right: auto; top: auto; border-radius: 0; box-shadow: none; border: none; border-top: 1px solid #DCE0E2; border-left: 5px solid #4299E0; }
</style>

<script>
(function () {
  const config = {
    mountId: "g3-mount",
    title: "Version-Pinned vs Alias-Based Loading",
    showAnnotationRail: true,
    hotspots: [
      {
        title: "Version-pinned",
        startLine: 1,
        endLine: 2,
        description: "Pinning to an explicit version number ties your application code to the current state of the registry. Every time a new version is promoted — dev to staging, or staging to production — every downstream consumer must be updated with the new number. A missed update means the wrong model is serving predictions."
      },
      {
        title: "Alias-based",
        startLine: 4,
        endLine: 5,
        description: "<code>@production</code> is a named pointer that MLflow resolves at load time. When a new model version is ready, operations teams move the alias — your application code does not change. This decoupling between version management and application code is the recommended pattern for production ML systems."
      }
    ],
    source: `# Version-pinned: requires a code change on every promotion.
model = mlflow.pyfunc.load_model(f"models:/{model_name}/3")

# Alias-based: the pointer moves — your code never changes.
model = mlflow.pyfunc.load_model(f"models:/{model_name}@production")`,
    showLineNumbers: true,
    displayStartLine: 1,
  };

  function escapeHtml(value) {
    return String(value)
      .replace(/&/g, "&amp;")
      .replace(/</g, "&lt;")
      .replace(/>/g, "&gt;");
  }

  function renderSourceExplorer(config) {
    const mount = document.getElementById(config.mountId);
    const lines = config.source.replace(/\n$/, "").split("\n");
    const start = config.startLine || 1;
    const end = config.endLine || lines.length;
    const visible = lines.slice(start - 1, end);
    const hotspots = config.hotspots || [];
    const displayStartLine = config.displayStartLine || start;

    function hotspotFor(lineNo) {
      const index = hotspots.findIndex(h => lineNo >= h.startLine && lineNo <= h.endLine);
      return index >= 0 ? { hotspot: hotspots[index], index } : null;
    }

    const codeHtml = visible.map((line, idx) => {
      const actualLineNo = start + idx;
      const displayLineNo = displayStartLine + idx;
      const hotspot = hotspotFor(actualLineNo);
      const classes = hotspot ? `sx-line sx-hotspot sx-hotspot-${hotspot.index}` : "sx-line";
      const attrs = hotspot ? `data-hotspot-index="${hotspot.index}"` : "";
      return `
        <div class="${classes} ${config.showLineNumbers ? "" : "no-lines"}" ${attrs}>
          ${config.showLineNumbers ? `<div class="sx-line-no">${displayLineNo}</div>` : ""}
          <div class="sx-line-code">${escapeHtml(line)}</div>
        </div>`;
    }).join("");

    const railHtml = config.showAnnotationRail && hotspots.length > 0
      ? `<div class="sx-rail">${hotspots.map((h, i) => `<button class="sx-chip" data-hotspot-index="${i}">${escapeHtml(h.title)}</button>`).join("")}</div>`
      : "";

    const overlayHtml = hotspots.length > 0
      ? `<div class="sx-overlay"><div class="sx-overlay-title"></div><div class="sx-overlay-body"></div></div>`
      : "";

    mount.innerHTML = `<div class="sx-wrap"><div class="sx-title">${escapeHtml(config.title)}</div>${railHtml}<div class="sx-code">${codeHtml}</div>${overlayHtml}</div>`;

    if (hotspots.length === 0) return;

    const overlay = mount.querySelector(".sx-overlay");
    const overlayTitle = mount.querySelector(".sx-overlay-title");
    const overlayBody = mount.querySelector(".sx-overlay-body");
    let activeHotspotIndex = null;

    function setActiveLines(idx) {
      mount.querySelectorAll(".sx-active-line").forEach(el => el.classList.remove("sx-active-line"));
      if (idx === null) return;
      mount.querySelectorAll(`.sx-hotspot-${idx}`).forEach(el => el.classList.add("sx-active-line"));
    }

    function setActiveChip(idx) {
      mount.querySelectorAll(".sx-chip").forEach(chip => {
        chip.classList.toggle("active", idx !== null && Number(chip.dataset.hotspotIndex) === idx);
      });
    }

    function setChipHover(idx, on) {
      mount.querySelectorAll(".sx-chip").forEach(chip => {
        chip.classList.toggle("sx-chip-hovered-from-line", on && Number(chip.dataset.hotspotIndex) === idx);
      });
    }

    function setLineHover(idx, on) {
      mount.querySelectorAll(`.sx-hotspot-${idx}`).forEach(line => line.classList.toggle("sx-hovered-from-chip", on));
    }

    function showHotspot(idx, triggerEl = null) {
      const hotspot = hotspots[idx];
      if (!hotspot) return;
      if (overlay.style.display === "block" && activeHotspotIndex === idx) {
        overlay.style.display = "none";
        activeHotspotIndex = null;
        setActiveChip(null);
        setActiveLines(null);
        return;
      }
      overlayTitle.textContent = hotspot.title;
      overlayBody.innerHTML = hotspot.description;
      overlay.style.display = "block";
      activeHotspotIndex = idx;
      setActiveChip(idx);
      setActiveLines(idx);
    }

    mount.querySelectorAll(".sx-hotspot").forEach(el => {
      const idx = Number(el.dataset.hotspotIndex);
      el.addEventListener("click", () => { showHotspot(idx, el); });
      el.addEventListener("mouseenter", () => { setLineHover(idx, true); setChipHover(idx, true); });
      el.addEventListener("mouseleave", () => {
        setLineHover(idx, false);
        setChipHover(idx, false);
        if (activeHotspotIndex !== null) { setActiveLines(activeHotspotIndex); }
      });
    });

    mount.querySelectorAll(".sx-chip").forEach(chip => {
      const idx = Number(chip.dataset.hotspotIndex);
      chip.addEventListener("click", () => {
        const associatedLine = mount.querySelector(`.sx-hotspot-${idx}`);
        showHotspot(idx, associatedLine || chip);
      });
      chip.addEventListener("mouseenter", () => { setLineHover(idx, true); });
      chip.addEventListener("mouseleave", () => {
        setLineHover(idx, false);
        if (activeHotspotIndex !== null) { setActiveLines(activeHotspotIndex); }
      });
    });

    mount.querySelector(".sx-wrap").addEventListener("click", (e) => {
      if (e.target.closest(".sx-hotspot")) return;
      if (e.target.closest(".sx-chip")) return;
      if (e.target.closest(".sx-overlay")) return;
      overlay.style.display = "none";
      activeHotspotIndex = null;
      setActiveChip(null);
      setActiveLines(null);
    });
  }

  renderSourceExplorer(config);
})();
</script>

In [0]:
# Version-pinned: requires a code change on every promotion.
model = mlflow.pyfunc.load_model(f"models:/{model_name}/1")
# Alias-based: the pointer moves — your code never changes.
model = mlflow.pyfunc.load_model(f"models:/{model_name}@dev")

### F4. Champion–Challenger Evaluation Loop

For A/B testing scenarios, the `champion` and `challenger` aliases drive a continuous evaluation cycle. A challenger is trained, compared against the current champion, and either promoted or retired — then the loop repeats.

<div style="max-width: 520px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 480 480" role="img" style="font-family: sans-serif;">
  <title>Champion-Challenger Evaluation Loop</title>
  <desc>Four stages arranged in a clockwise loop: Train Challenger, Compare vs Champion, Promote or Retire, and Monitor Champion, repeating continuously.</desc>
  <defs>
    <marker id="cyc-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="#5E7077" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker>
  </defs>
  <path d="M 315 110 A 150 150 0 0 1 370 165" fill="none" stroke="#5E7077" stroke-width="2" marker-end="url(#cyc-arrow)"/>
  <path d="M 370 315 A 150 150 0 0 1 315 370" fill="none" stroke="#5E7077" stroke-width="2" marker-end="url(#cyc-arrow)"/>
  <path d="M 165 370 A 150 150 0 0 1 110 315" fill="none" stroke="#5E7077" stroke-width="2" marker-end="url(#cyc-arrow)"/>
  <path d="M 110 165 A 150 150 0 0 1 165 110" fill="none" stroke="#5E7077" stroke-width="2" marker-end="url(#cyc-arrow)"/>
  <rect x="168" y="58" width="144" height="64" rx="8" fill="#4299E0" fill-opacity="0.12" stroke="#4299E0" stroke-width="1.5"/>
  <text text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026"><tspan x="240" y="83">Train</tspan><tspan x="240" dy="18">Challenger</tspan></text>
  <rect x="318" y="208" width="144" height="64" rx="8" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1.5"/>
  <text x="390" y="245" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Compare</text>
  <rect x="168" y="358" width="144" height="64" rx="8" fill="#FFAB00" fill-opacity="0.15" stroke="#FFAB00" stroke-width="1.5"/>
  <text text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026"><tspan x="240" y="383">Promote/</tspan><tspan x="240" dy="18">Retire</tspan></text>
  <rect x="18" y="208" width="144" height="64" rx="8" fill="#FF5F46" fill-opacity="0.10" stroke="#FF5F46" stroke-width="1.5"/>
  <text text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026"><tspan x="90" y="233">Monitor</tspan><tspan x="90" dy="18">Champion</tspan></text>
  <text x="240" y="246" text-anchor="middle" font-size="14" font-weight="700" fill="#5E7077">Repeat</text>
</svg>
</div>


## Conclusion

In this demonstration, you walked through the model development lifecycle using MLflow and bakehouse sales data:

1. Prepared features using SQL from bakehouse sales transactions.
2. Trained and logged a random forest model using MLflow.
3. Tracked the experiment and reviewed run details in the MLflow UI.
4. Registered the model in Unity Catalog.
5. Applied a model version alias (`dev`) to manage deployment stages.

### Next Steps

With the model registered and aliased, you can compare additional experiment runs, promote versions through `staging` and `production` aliases, and deploy the model to a serving endpoint for real-time predictions.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>